<a href="https://colab.research.google.com/github/avinashsk112-creator/GEC/blob/main/BPNT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1: Install Dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q faster-whisper scenedetect opencv-python pydantic ipywidgets ultralytics scipy
print("[✓] Dependencies installed successfully.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 13.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.0/35.0 MB 71.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 MB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.9/96.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 83.0 MB/s eta 0:00:00
[✓] Dependencies installed

In [2]:
# Cell 2: Setup Paths & Face Model
import os
import json
import csv
import cv2
import numpy as np
import subprocess
import torch
from pathlib import Path
from scenedetect import open_video, SceneManager
from scenedetect.detectors import ContentDetector
from faster_whisper import WhisperModel
from pydantic import BaseModel
from typing import List
from ultralytics import YOLO
from scipy.ndimage import gaussian_filter1d

# Setup target delivery directories
BASE_DIR = Path("/content/output_delivery")
CLIPS_DIR = BASE_DIR / "clips"
META_DIR = BASE_DIR / "metadata"
THUMB_DIR = BASE_DIR / "thumbnails"

for p in [BASE_DIR, CLIPS_DIR, META_DIR, THUMB_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Load lightweight face model via PyTorch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[*] Loading Face Model on {device.upper()}...")
face_model = YOLO("yolov8n-face.pt") if os.path.exists("yolov8n-face.pt") else YOLO("yolov8n.pt")

print("[✓] Directories initialized and PyTorch Face Detector loaded.")

/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.13/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
/usr/local/lib/python3.13/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.
[*] Loading Face Model on CPU...
[✓] Directories initialized and PyTorch Face Detector loaded.


In [3]:
# Cell 3: Shot & Cut Boundary Detection
def detect_visual_scenes(video_path: str, threshold: float = 27.0):
    print(f"[*] Analyzing visual cuts for: {os.path.basename(video_path)}")
    video = open_video(video_path)
    scene_manager = SceneManager()
    scene_manager.add_detector(ContentDetector(threshold=threshold))
    scene_manager.detect_scenes(video)

    scene_list = scene_manager.get_scene_list()
    cut_timestamps = []
    for scene in scene_list:
        start_sec = scene[0].seconds
        end_sec = scene[1].seconds
        cut_timestamps.append({"start": round(start_sec, 2), "end": round(end_sec, 2)})

    print(f"[✓] Detected {len(cut_timestamps)} physical cuts.")
    return cut_timestamps

In [15]:
# Cell 4: Audio Extraction & Transcription (PyAV-Bypass via Raw PCM)
import os
import subprocess
import torch
import numpy as np
import logging
import warnings
from faster_whisper import WhisperModel

# Suppress Hugging Face warnings
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*unauthenticated requests to the HF Hub.*")

def extract_and_transcribe(video_path: str):
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    comp_type = "float16" if dev == "cuda" else "int8"
    print(f"[*] Loading Faster-Whisper on {dev.upper()} ({comp_type})...")

    # 1. Use system ffmpeg to decode directly to raw 16-bit 16kHz mono PCM stream
    print("[*] Decoding audio stream via system FFmpeg...")
    cmd = [
        "ffmpeg", "-nostdin", "-threads", "0", "-i", video_path,
        "-f", "s16le", "-ac", "1", "-acodec", "pcm_s16le", "-ar", "16000", "-"
    ]
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    raw_audio, _ = process.communicate()

    # 2. Convert raw PCM bytes to normalized float32 array (Whisper native format)
    audio_np = np.frombuffer(raw_audio, np.int16).flatten().astype(np.float32) / 32768.0
    total_duration = round(len(audio_np) / 16000.0, 2)
    print(f"[*] Total audio duration: {total_duration}s (~{round(total_duration/60, 1)} mins)")

    # 3. Transcribe NumPy array directly (completely bypasses av.open())
    model = WhisperModel("small", device=dev, compute_type=comp_type)
    print("[*] Transcribing dialogue chunks...")
    segments, info = model.transcribe(audio_np, beam_size=1, word_timestamps=True)

    transcript_blocks = []
    last_printed_sec = 0

    for segment in segments:
        duration = max(0.1, segment.end - segment.start)
        word_count = len(segment.text.split())
        wps = word_count / duration

        block = {
            "start": round(segment.start, 2),
            "end": round(segment.end, 2),
            "text": segment.text.strip(),
            "wps": round(wps, 2),
            "words": word_count
        }
        transcript_blocks.append(block)

        if segment.end - last_printed_sec >= 30:
            pct = min(100.0, round((segment.end / total_duration) * 100, 1))
            print(f"    --> Transcribed up to {round(segment.end, 1)}s / {total_duration}s ({pct}%)")
            last_printed_sec = segment.end

    print(f"[✓] Transcription complete: {len(transcript_blocks)} dialogue blocks indexed.")
    return transcript_blocks

In [5]:
# Cell 5: Editorial Decision Engine (Local Scoring)
class ClipMetadata(BaseModel):
    title: str
    category: str
    start_time: float
    end_time: float
    dramatic_rationale: str
    recommended_thumbnail_time: float

def analyze_narrative_locally(transcript_blocks: list, cuts: list) -> list:
    print("[*] Computing narrative density, pacing, and cut alignment...")
    if not transcript_blocks:
        print("[!] No speech detected.")
        return []

    total_duration = transcript_blocks[-1]["end"]
    selected_clips = []

    def snap_to_cut(timestamp, max_delta=3.0):
        for c in cuts:
            if abs(c["start"] - timestamp) <= max_delta:
                return round(c["start"], 2)
        return round(timestamp, 2)

    # 1. Best Scene Selection (Target: 120s - 180s)
    best_scene_target = 150.0
    best_scene_score = -1
    best_scene_window = (0.0, best_scene_target)

    for i in range(len(transcript_blocks)):
        start = transcript_blocks[i]["start"]
        end_limit = start + best_scene_target
        if end_limit > total_duration:
            break

        window_blocks = [b for b in transcript_blocks if b["start"] >= start and b["end"] <= end_limit]
        total_words = sum(b["words"] for b in window_blocks)

        if total_words > best_scene_score:
            best_scene_score = total_words
            actual_end = window_blocks[-1]["end"] if window_blocks else end_limit
            best_scene_window = (start, min(start + 180.0, max(start + 120.0, actual_end)))

    bs_start = snap_to_cut(best_scene_window[0])
    bs_end = snap_to_cut(best_scene_window[1])

    selected_clips.append(ClipMetadata(
        title="Dramatic Narrative Peak",
        category="Best_Scene",
        start_time=bs_start,
        end_time=bs_end,
        dramatic_rationale=f"Highest continuous dialogue density ({best_scene_score} words) reflecting central conflict.",
        recommended_thumbnail_time=round(bs_start + 10.0, 2)
    ))

    # 2. Viral Reels Selection (Target: 50s - 60s each)
    reel_candidates = []
    reel_target = 55.0

    for i in range(len(transcript_blocks)):
        start = transcript_blocks[i]["start"]
        end_limit = start + reel_target
        if end_limit > total_duration:
            break
        if not (end_limit < bs_start or start > bs_end):
            continue

        window_blocks = [b for b in transcript_blocks if b["start"] >= start and b["end"] <= end_limit]
        avg_wps = np.mean([b["wps"] for b in window_blocks]) if window_blocks else 0
        total_words = sum(b["words"] for b in window_blocks)
        score = avg_wps * total_words

        reel_candidates.append((score, start, start + reel_target))

    reel_candidates.sort(reverse=True, key=lambda x: x[0])
    chosen_reels = []

    for cand in reel_candidates:
        if len(chosen_reels) >= 2:
            break
        c_start, c_end = cand[1], cand[2]
        collision = any(not (c_end < cr[0] or c_start > cr[1]) for cr in chosen_reels)
        if not collision:
            chosen_reels.append((c_start, c_end))

    for idx, (r_start, r_end) in enumerate(chosen_reels):
        snap_s = snap_to_cut(r_start)
        snap_e = snap_to_cut(r_end)
        selected_clips.append(ClipMetadata(
            title=f"Viral Dialogue Hook {idx+1}",
            category="Reel",
            start_time=snap_s,
            end_time=snap_e,
            dramatic_rationale="High conversational pace and dramatic exchange suited for vertical virality.",
            recommended_thumbnail_time=round(snap_s + 5.0, 2)
        ))

    # 3. Webisode Story Beats (4 sequential acts)
    act_duration = total_duration / 4.0
    for act in range(4):
        act_center = (act * act_duration) + (act_duration / 2.0)
        b_start = snap_to_cut(max(0.0, act_center - 35.0))
        b_end = snap_to_cut(min(total_duration, act_center + 35.0))

        selected_clips.append(ClipMetadata(
            title=f"Webisode Story Beat {act+1}",
            category="Webisode_Beat",
            start_time=b_start,
            end_time=b_end,
            dramatic_rationale=f"Chronological story anchor representing episode quarter {act+1}.",
            recommended_thumbnail_time=round(b_start + 8.0, 2)
        ))

    print(f"[✓] Editorial selection complete: {len(selected_clips)} assets queued.")
    return selected_clips

In [19]:
# Cell 6: Broadcast Body & Head-Weighted Auto-Reframe Engine
import cv2
import numpy as np
import subprocess
import os
from scipy.ndimage import gaussian_filter1d

class BroadcastSmartReframe:
    def __init__(self, target_ratio="9:16"):
        self.target_ratio = target_ratio

    def _compute_target_dims(self, width, height):
        if self.target_ratio == "9:16":
            target_w = int(height * (9 / 16))
            target_h = height
        elif self.target_ratio == "1:1":
            target_w = height
            target_h = height
        else:
            target_w = width
            target_h = height
        if target_w % 2 != 0:
            target_w -= 1
        return target_w, target_h

    def _detect_shot_boundaries(self, cap, threshold=28.0):
        """Detect hard camera cuts so framing never slides across cuts."""
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        shot_boundaries = [0]
        prev_gray = None
        frame_idx = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            small = cv2.resize(frame, (320, 180))
            gray = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
            if prev_gray is not None:
                diff = np.mean(cv2.absdiff(gray, prev_gray))
                if diff > threshold:
                    shot_boundaries.append(frame_idx)
            prev_gray = gray
            frame_idx += 1

        shot_boundaries.append(frame_idx)
        return shot_boundaries, frame_idx

    def _pass1_person_tracking(self, cap, total_frames, width, height, target_w):
        """Pass 1: Detect persons (full torso + head) and calculate broadcast-safe X centers."""
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        raw_centers = np.full(total_frames, width // 2, dtype=np.float32)

        last_valid_cx = width // 2
        active_speaker_x = width // 2
        prev_gray_mouth = None

        for f_idx in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break

            # Predict class 0 (Person) - detects head, shoulders, torso even if facing away
            results = person_model.predict(
                source=frame, classes=[0], conf=0.30, verbose=False, device=device
            )

            persons = []
            for r in results:
                for box in r.boxes:
                    coords = box.xyxy[0].cpu().numpy().astype(int)
                    x1, y1, x2, y2 = coords
                    x1, y1 = max(0, x1), max(0, y1)
                    x2, y2 = min(width, x2), min(height, y2)
                    pw, ph = x2 - x1, y2 - y1

                    # Ignore tiny background extras
                    if pw > 40 and ph > 80:
                        # Character center of mass (weighted toward chest/head: upper 40%)
                        char_cx = x1 + (pw // 2)

                        # Mouth/head region activity
                        head_y2 = y1 + int(ph * 0.35)
                        head_roi = frame[y1:head_y2, x1:x2]

                        persons.append({
                            "cx": char_cx,
                            "box": (x1, y1, pw, ph),
                            "area": pw * ph,
                            "width": pw
                        })

            if len(persons) == 0:
                # Hold previous framing smoothly if no one detected
                raw_centers[f_idx] = last_valid_cx

            elif len(persons) == 1:
                # Single character in frame: Lock directly on character
                target_x = persons[0]["cx"]
                raw_centers[f_idx] = target_x
                last_valid_cx = target_x

            else:
                # Multiple characters (e.g. Dialogue Scene / Two-Shot)
                persons.sort(key=lambda p: p["area"], reverse=True)
                p1, p2 = persons[0], persons[1]

                # Check distance between two main characters
                min_char_x = min(p1["cx"], p2["cx"])
                max_char_x = max(p1["cx"], p2["cx"])
                span = max_char_x - min_char_x

                # If both characters fit inside the 9:16 frame width comfortably:
                if span < (target_w * 0.75):
                    # Center both actors perfectly
                    group_center = (min_char_x + max_char_x) // 2
                    raw_centers[f_idx] = group_center
                    last_valid_cx = group_center
                else:
                    # Characters are too far apart for single 9:16 frame:
                    # Lock firmly on the dominant/foreground subject (largest area)
                    # to prevent ping-ponging across the room
                    dominant_cx = p1["cx"]
                    raw_centers[f_idx] = dominant_cx
                    last_valid_cx = dominant_cx

        return raw_centers

    def _pass2_cinematic_smoothing(self, raw_centers, shot_boundaries, total_frames, target_w, width):
        """Pass 2: Apply temporal Gaussian smoothing per shot so movement is butter-smooth."""
        smooth_x = np.copy(raw_centers)
        half_w = target_w // 2

        for i in range(len(shot_boundaries) - 1):
            s_start = shot_boundaries[i]
            s_end = shot_boundaries[i + 1]
            shot_len = s_end - s_start

            if shot_len <= 1:
                continue

            shot_segment = raw_centers[s_start:s_end]

            # For short shots (< 1.5 sec), lock to median center to avoid nervous panning
            if shot_len < 40:
                median_x = np.median(shot_segment)
                filtered = np.full(shot_len, median_x)
            else:
                # Apply heavy Gaussian temporal filter for Steadicam feel
                sigma = min(25.0, shot_len / 4.0)
                filtered = gaussian_filter1d(shot_segment, sigma=sigma, mode="nearest")

            # Clamp strictly within 16:9 frame boundaries
            filtered = np.clip(filtered, half_w, width - half_w)
            smooth_x[s_start:s_end] = filtered

        return smooth_x.astype(int)

    def process(self, input_clip: str, output_path: str):
        cap = cv2.VideoCapture(input_clip)
        width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = cap.get(cv2.CAP_PROP_FPS) or 25.0

        target_w, target_h = self._compute_target_dims(width, height)

        # 1. Shot boundary cuts
        shot_boundaries, total_frames = self._detect_shot_boundaries(cap)

        # 2. Pass 1: Person body & torso tracking
        raw_centers = self._pass1_person_tracking(cap, total_frames, width, height, target_w)

        # 3. Pass 2: Cinematic smoothing with shot cut locks
        final_centers = self._pass2_cinematic_smoothing(raw_centers, shot_boundaries, total_frames, target_w, width)

        # 4. Render vertical/square raster
        temp_crop_video = output_path.replace(".mp4", "_temp.mp4")
        out = cv2.VideoWriter(temp_crop_video, cv2.VideoWriter_fourcc(*'mp4v'), fps, (target_w, target_h))

        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
        half_w = target_w // 2

        for f_idx in range(total_frames):
            ret, frame = cap.read()
            if not ret:
                break

            cx = final_centers[f_idx]
            x1 = max(0, min(cx - half_w, width - target_w))
            cropped = frame[:, x1:x1 + target_w]
            out.write(cropped)

        cap.release()
        out.release()

        # 5. Remux normalized audio
        cmd = (
            f"ffmpeg -y -i '{temp_crop_video}' -i '{input_clip}' "
            f"-map 0:v:0 -map 1:a:0 -c:v libx264 -pix_fmt yuv420p -c:a copy '{output_path}' -loglevel error"
        )
        subprocess.run(cmd, shell=True)
        if os.path.exists(temp_crop_video):
            os.remove(temp_crop_video)


# Unified Render Functions
def render_smart_crop(input_clip: str, output_path: str, target_ratio: str = "9:16"):
    reframe_engine = BroadcastSmartReframe(target_ratio=target_ratio)
    reframe_engine.process(input_clip, output_path)

def cut_and_normalize_clip(source_video: str, start_time: float, end_time: float, output_16_9: str):
    duration = end_time - start_time
    fade_d = min(0.5, duration / 4.0)
    vf_filter = f"fade=t=in:st=0:d={fade_d},fade=t=out:st={duration-fade_d}:d={fade_d}"
    af_filter = f"afade=t=in:st=0:d={fade_d},afade=t=out:st={duration-fade_d}:d={fade_d},loudnorm=I=-24:LRA=11:TP=-1.0"

    cmd = (
        f"ffmpeg -y -ss {start_time} -to {end_time} -i '{source_video}' "
        f"-vf '{vf_filter}' -af '{af_filter}' "
        f"-c:v libx264 -preset fast -crf 20 -c:a aac -b:a 192k '{output_16_9}' -loglevel error"
    )
    subprocess.run(cmd, shell=True)

In [7]:
# Cell 7: Master Execution Pipeline
def run_pipeline(source_video_path: str):
    print(f"\n================ STARTING PIPELINE: {os.path.basename(source_video_path)} ================")

    # 1. Physical scene cut detection
    cuts = detect_visual_scenes(source_video_path)

    # 2. Local speech indexing
    transcript_blocks = extract_and_transcribe(source_video_path)

    # 3. Local editorial selection
    selected_clips = analyze_narrative_locally(transcript_blocks, cuts)

    # 4. Export JSON & CSV Metadata Deliverables
    json_path = META_DIR / "delivery_manifest.json"
    csv_path = META_DIR / "delivery_manifest.csv"

    with open(json_path, "w") as jf:
        json.dump([c.model_dump() for c in selected_clips], jf, indent=2)

    with open(csv_path, "w", newline="") as cf:
        writer = csv.writer(cf)
        writer.writerow(["Title", "Category", "Start_Sec", "End_Sec", "Duration", "Rationale"])
        for c in selected_clips:
            writer.writerow([c.title, c.category, c.start_time, c.end_time, round(c.end_time - c.start_time, 2), c.dramatic_rationale])

    print(f"[✓] Metadata deliverables written to {META_DIR}")

    # 5. Render Video Cuts, Crops & Thumbnails
    for idx, clip in enumerate(selected_clips):
        prefix = f"{idx+1}_{clip.category}_{int(clip.start_time)}s"
        base_16_9 = str(CLIPS_DIR / f"{prefix}_16x9.mp4")
        crop_9_16 = str(CLIPS_DIR / f"{prefix}_9x16.mp4")
        crop_1_1 = str(CLIPS_DIR / f"{prefix}_1x1.mp4")
        thumb_1 = str(THUMB_DIR / f"{prefix}_thumb1.jpg")
        thumb_2 = str(THUMB_DIR / f"{prefix}_thumb2.jpg")

        print(f"[*] Rendering [{clip.category}]: {clip.title} ({clip.start_time}s -> {clip.end_time}s)")

        # 16:9 master with EBU R128 (-24 LKFS)
        cut_and_normalize_clip(source_video_path, clip.start_time, clip.end_time, base_16_9)

        # Active speaker re-framing for Reels
        if clip.category == "Reel":
            print("  --> Active Speaker tracking: rendering 1:1 and 9:16 crops...")
            render_smart_crop(base_16_9, crop_1_1, target_ratio="1:1")
            render_smart_crop(base_16_9, crop_9_16, target_ratio="9:16")

        # Two thumbnails per deliverable
        t1 = clip.recommended_thumbnail_time
        t2 = min(clip.start_time + 4.0, clip.end_time)
        subprocess.run(f"ffmpeg -y -ss {t1} -i '{source_video_path}' -vframes 1 -q:v 2 '{thumb_1}' -loglevel error", shell=True)
        subprocess.run(f"ffmpeg -y -ss {t2} -i '{source_video_path}' -vframes 1 -q:v 2 '{thumb_2}' -loglevel error", shell=True)

    print("\n================ PIPELINE EXECUTION FINISHED ================")
    print(f"All assets generated under: {BASE_DIR}")

In [12]:
# Cell 8: Upload Button UI
from google.colab import files
import ipywidgets as widgets
from IPython.display import display

upload_btn = widgets.Button(
    description="Upload Episode Video",
    button_style="primary",
    icon="upload",
    tooltip="Upload MP4/MOV/MKV episode to process"
)

status_output = widgets.Output()

def on_button_clicked(b):
    with status_output:
        status_output.clear_output()
        print("Select your video file from the prompt below:")

        uploaded = files.upload()
        if not uploaded:
            print("[!] Upload cancelled.")
            return

        for filename in uploaded.keys():
            video_path = os.path.abspath(filename)
            size_mb = os.path.getsize(video_path) / (1024 * 1024)
            print(f"\n[✓] Uploaded: {video_path} ({size_mb:.2f} MB)")
            print("[*] Running pipeline...\n")

            try:
                run_pipeline(video_path)
            except Exception as e:
                print(f"\n[!] Error encountered: {e}")

upload_btn.on_click(on_button_clicked)

print("Click below to upload your episode and run the pipeline:")
display(upload_btn, status_output)

Click below to upload your episode and run the pipeline:


Button(button_style='primary', description='Upload Episode Video', icon='upload', style=ButtonStyle(), tooltip…

Output()

In [21]:
# Cell 9: Package & Download Deliverables
from google.colab import files
import shutil

zip_target = "/content/episode_delivery_package.zip"
print("[*] Archiving output deliverables...")
shutil.make_archive("/content/episode_delivery_package", 'zip', "/content/output_delivery")

print(f"[✓] Archive created: {zip_target}")
print("[*] Initiating browser download...")
files.download(zip_target)

[*] Archiving output deliverables...
[✓] Archive created: /content/episode_delivery_package.zip
[*] Initiating browser download...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
# Cell 1 Fix: Pin PyAV and install dependencies
!apt-get update -qq && apt-get install -y -qq ffmpeg libsndfile1
!pip install -q "av<14" faster-whisper scenedetect opencv-python pydantic ipywidgets ultralytics scipy
print("[✓] Dependencies installed with compatible PyAV version.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.1/34.1 MB 64.3 MB/s eta 0:00:00
[✓] Dependencies installed with compatible PyAV version.


In [13]:
# Fix Cell: Downgrade PyAV to a compatible version
!pip install -q "av<14" --force-reinstall
!pip install -q --no-deps --force-reinstall git+https://github.com/SYSTRAN/faster-whisper.git
print("[✓] PyAV compatibility fix installed.")


  Preparing metadata (setup.py) ... done
[✓] PyAV compatibility fix installed.


In [16]:
# Resume execution
import glob

videos = glob.glob("/content/*.mp4") + glob.glob("/content/*.mkv") + glob.glob("/content/*.mov")
source_videos = [v for v in videos if "output_delivery" not in v and "_temp" not in v]

if source_videos:
    latest_video = source_videos[0]
    print(f"[*] Resuming pipeline on: {latest_video}")
    run_pipeline(latest_video)
else:
    print("[!] No uploaded episode found. Run Cell 8 to upload your video.")

INFO:pyscenedetect:Detecting scenes...


[*] Resuming pipeline on: /content/tester 2 (1).mp4

================ STARTING PIPELINE: tester 2 (1).mp4 ================
[*] Analyzing visual cuts for: tester 2 (1).mp4
[✓] Detected 314 physical cuts.
[*] Loading Faster-Whisper on CPU (int8)...
[*] Decoding audio stream via system FFmpeg...
[*] Total audio duration: 749.93s (~12.5 mins)
[*] Transcribing dialogue chunks...
    --> Transcribed up to 30.0s / 749.93s (4.0%)
    --> Transcribed up to 63.9s / 749.93s (8.5%)
    --> Transcribed up to 94.0s / 749.93s (12.5%)
    --> Transcribed up to 127.9s / 749.93s (17.1%)
    --> Transcribed up to 158.2s / 749.93s (21.1%)
    --> Transcribed up to 192.7s / 749.93s (25.7%)
    --> Transcribed up to 223.5s / 749.93s (29.8%)
    --> Transcribed up to 253.9s / 749.93s (33.9%)
    --> Transcribed up to 284.0s / 749.93s (37.9%)
    --> Transcribed up to 314.2s / 749.93s (41.9%)
    --> Transcribed up to 346.5s / 749.93s (46.2%)
    --> Transcribed up to 376.7s / 749.93s (50.2%)
    --> Transcri

In [18]:
# Cell: Verify YOLOv8 Person/Pose weights
from ultralytics import YOLO
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
# Using yolov8n.pt natively detects class 0: 'person' (full torso + head)
person_model = YOLO("yolov8n.pt")
print(f"[✓] YOLO Person Detector loaded on {device.upper()}")

[✓] YOLO Person Detector loaded on CPU


In [20]:
# Re-render with Torso & Group-Aware Framing
import glob

videos = glob.glob("/content/*.mp4") + glob.glob("/content/*.mkv") + glob.glob("/content/*.mov")
source_videos = [v for v in videos if "output_delivery" not in v and "_temp" not in v]

if source_videos:
    latest_video = source_videos[0]
    print(f"[*] Re-rendering with Person Torso & Broadcast Safe Reframe: {latest_video}")
    run_pipeline(latest_video)
else:
    print("[!] No source video found in /content. Run Cell 8 to upload.")

INFO:pyscenedetect:Detecting scenes...


[*] Re-rendering with Person Torso & Broadcast Safe Reframe: /content/tester 2 (1).mp4

================ STARTING PIPELINE: tester 2 (1).mp4 ================
[*] Analyzing visual cuts for: tester 2 (1).mp4
[✓] Detected 314 physical cuts.
[*] Loading Faster-Whisper on CPU (int8)...
[*] Decoding audio stream via system FFmpeg...
[*] Total audio duration: 749.93s (~12.5 mins)
[*] Transcribing dialogue chunks...
    --> Transcribed up to 30.0s / 749.93s (4.0%)
    --> Transcribed up to 63.9s / 749.93s (8.5%)
    --> Transcribed up to 94.0s / 749.93s (12.5%)
    --> Transcribed up to 127.9s / 749.93s (17.1%)
    --> Transcribed up to 158.2s / 749.93s (21.1%)
    --> Transcribed up to 192.7s / 749.93s (25.7%)
    --> Transcribed up to 223.5s / 749.93s (29.8%)
    --> Transcribed up to 254.0s / 749.93s (33.9%)
    --> Transcribed up to 284.0s / 749.93s (37.9%)
    --> Transcribed up to 314.2s / 749.93s (41.9%)
    --> Transcribed up to 346.5s / 749.93s (46.2%)
    --> Transcribed up to 376.7s